# 📓 MTG-Jamendo → parsed dataset (Google Colab + Drive)

The entire pipeline processes raw data on **Google Drive**, while writing `.npz` files to a **local disk** (for speed), followed by a secure upload to Drive after each batch.

**Sequence:** mount → config → deps → MTG metadata → `create_labels` → download audio → smoke test → supervisor → `create_labels --check` → final sync → (optional) delete MP3s.

⚠️ If the runtime is disabled, re-run the **config** cell first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys

DRIVE_REPO = Path("/content/drive/MyDrive/music-analyzer_v2")
LOCAL_PARSED = Path("/content/parsed_dataset")
MTG_REPO = DRIVE_REPO / "mtg-jamendo-dataset"
DRIVE_PARSED = DRIVE_REPO / "parsed_dataset"

if not DRIVE_REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ChristosPaterakisntua/music-analyzer-v2.git",
         str(DRIVE_REPO)], check=True)

LOCAL_PARSED.mkdir(parents=True, exist_ok=True)

# environment read by music_analyzer_v2.utils (BEFORE any import of the package)
os.environ["MA2_RAW_DATASET"] = str(MTG_REPO)
os.environ["MA2_PARSED_DATASET"] = str(LOCAL_PARSED)
os.environ["MA2_CHECKPOINT_DIR"] = str(DRIVE_REPO / "checkpoints")
os.environ["MA2_SAVED_MODELS_DIR"] = str(DRIVE_REPO / "saved_models")
os.environ["PYTHONPATH"] = str(DRIVE_REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, str(DRIVE_REPO / "src"))

print("repo     :", DRIVE_REPO)
print("raw audio:", MTG_REPO / "dataset")
print("npz dir  :", LOCAL_PARSED)
print("upload to:", DRIVE_PARSED)

In [ ]:
# torch is already pre-installed in Colab — we don't need to install it again
%pip install -q librosa matplotlib pandas scikit-learn

In [ ]:
# MTG metadata (data/splits/split-0/*.tsv — MB only).
# The downloader fetches these from the official GitHub repo (they are not on the audio CDN).
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py --mode splits --dest {MTG_REPO}


In [ ]:
# labels.json -> LOCAL_PARSED (used as a manifest by the downloader)
completed = subprocess.run([sys.executable, "-m", "music_analyzer_v2.create_labels"])
assert completed.returncode == 0, "create_labels απέτυχε"
print("labels:", LOCAL_PARSED / "labels.json")

In [ ]:
# What will be downloaded (a subset from labels.json, quality=low).
# NOTE: Always from the universal raw_30s source: https://cdn.freesound.org/mtg-jamendo/raw_30s/audio-low/<NN>/<id>.low.mp3
# The per-subset paths (autotagging_genre/..., autotagging_moodtheme/...) return a 404
# for tracks that do not belong to that specific subset.
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py --dry-run

In [ ]:
# test: 10 files
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py --limit 10

In [ ]:
# The entire subset (~33 GB, 8,457 tracks) is on Drive.
# Interrupted? Just run it again — completed items are skipped,
# and partially downloaded ones will resume (Range resume).
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py

In [ ]:
# smoke test: export 3 tracks to verify the entire pipeline (paths, npz, validation)
import music_analyzer_v2.create_dataset as create_dataset_module
create_dataset_module.create_parsed_dataset(max_tracks=3, num_workers=2)

In [ ]:
# watchdog: run create_dataset in chunks; after each chunk the new .npz are
# uploaded to Drive and the local copies deleted (frees the ephemeral /content
# disk). If the session dies -> rerun the same cell to continue (resumable).
completed = subprocess.run([
    sys.executable, str(DRIVE_REPO / "scripts" / "supervised_create_dataset.py"),
    "--upload-to", str(DRIVE_PARSED),
    "--max-tracks", "500",
    "--delete-local-npz",
])
print("supervisor exit code:", completed.returncode)

In [ ]:
# final labels.json: only tracks with a valid .npz (+ deletion of bad .npz files)
completed = subprocess.run([sys.executable, "-m", "music_analyzer_v2.create_labels",
                            "--check", "--delete"])
assert completed.returncode == 0, "create_labels --check απέτυχε"

In [ ]:
# final sweep: upload labels.json + anything else remaining, and verify against Drive
!{sys.executable} {DRIVE_REPO}/scripts/supervised_create_dataset.py --upload-only --upload-to {DRIVE_PARSED}

In [ ]:
# OPTIONAL: Free up ~45 GB on the Drive (delete the parsed MP3s).
# Test with a dry-run first; then uncomment --confirm.
!{sys.executable} {DRIVE_REPO}/scripts/delete_parsed_mp3.py
# !{sys.executable} {DRIVE_REPO}/scripts/delete_parsed_mp3.py --confirm

In [ ]:
# NEW SESSION (e.g., for training): brings the .npz files from Drive back to the local disk
import shutil
LOCAL_PARSED.mkdir(parents=True, exist_ok=True)
restored = 0
for source in sorted(DRIVE_PARSED.glob("*.npz")):
    target = LOCAL_PARSED / source.name
    if target.exists() and target.stat().st_size == source.stat().st_size:
        continue
    shutil.copy2(source, target)
    restored += 1
labels = DRIVE_PARSED / "labels.json"
if labels.exists():
    shutil.copy2(labels, LOCAL_PARSED / "labels.json")
print(f"restored {restored} .npz -> {LOCAL_PARSED}")

### Notes

- **Resume functionality:** The downloader, `create_dataset` (skipping existing `.npz` files), and the supervisor all support resuming from where they left off.
- **Audio source:** Use only the universal `raw_30s` source (`.../raw_30s/audio-low/<NN>/<id>.low.mp3`). Note that per-subset URLs (e.g., `autotagging_genre/...`, `autotagging_moodtheme/...`) return a **404** error for tracks not belonging to that specific subset, even though the corresponding file exists within `raw_30s`.
- **Dataset size:** `labels.json` contains 8,457 tracks (train: 4,742 / validation: 1,799 / test: 1,916)—this includes only tracks that have *both* a genre and a feeling/mood present in the target label set.
- **Drive space (~200 GB):** ~33 GB of audio + `.npz` files (~65 GB) + the repository—there is plenty of room. You can delete the MP3 files after they have been parsed (using `delete_parsed_mp3.py --confirm`).
- **Training:** Run `handle_training.ipynb` after the configuration cell (ensure `MA2_PARSED_DATASET` points to a local path for faster I/O). Checkpoints are uploaded to Google Drive via `MA2_CHECKPOINT_DIR`; if you prefer to keep them locally, update this setting in the configuration.
- The legacy C++ supervisor (`scripts/supervised_create_dataset.cpp`) is retained for Windows compatibility; in this environment, the Python-based version (`supervised_create_dataset.py`) is used.